# Practice Dataset Statistics

Notebook này chỉ tập trung vào `Practice_Dataset` vì đây là phần có đủ ground truth driver để:

- thống kê phân bố label
- xem phân bố theo trip và theo subject
- phân tích alertness, transition và segment
- chuẩn bị training index / grouped validation về sau

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "AGENTS.md").exists():
    if (PROJECT_ROOT.parent / "AGENTS.md").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
    else:
        raise FileNotFoundError("Cannot locate project root from the current notebook working directory.")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ml.notebooks import fleetiq_notebook_utils as nb

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

In [ ]:
practice_df = nb.build_practice_frame_index()
practice_df.head()

In [ ]:
dist = nb.state_distribution_tables(practice_df)
dist["overall"]

In [ ]:
dist["by_trip"]

In [ ]:
dist["by_subject"]

In [ ]:
alertness_stats = (
    practice_df.groupby("driver_state")["alertness_score"]
    .agg(["count", "mean", "std", "min", "median", "max"])
    .sort_values("mean")
)
alertness_stats

In [ ]:
transition_matrix = nb.compute_state_transitions(practice_df)
transition_pivot = transition_matrix.pivot(
    index="driver_state",
    columns="next_state",
    values="count",
).fillna(0).astype(int)
transition_pivot

In [ ]:
trip_id = "T01-Sample"
trip_df = practice_df.query("trip_id == @trip_id").copy()
segments = nb.derive_state_segments(trip_df)
segments

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

state_order = ["alert", "drowsy", "yawning", "distracted", "microsleep"]
state_map = {state: idx for idx, state in enumerate(state_order)}

axes[0].plot(trip_df["timestamp"], trip_df["alertness_score"], color="#F37021", linewidth=2)
axes[0].set_ylabel("alertness_score")
axes[0].set_title(f"Alertness timeline | {trip_id}")

axes[1].step(
    trip_df["timestamp"],
    trip_df["driver_state"].map(state_map),
    where="post",
    color="#19226D",
    linewidth=2,
)
axes[1].set_yticks(list(state_map.values()))
axes[1].set_yticklabels(state_order)
axes[1].set_ylabel("driver_state")
axes[1].set_xlabel("timestamp (s)")
axes[1].set_title(f"State timeline | {trip_id}")

plt.tight_layout()

In [ ]:
out_dir = PROJECT_ROOT / "artifacts" / "practice_stats"
out_dir.mkdir(parents=True, exist_ok=True)
practice_df.to_csv(out_dir / "practice_frame_index.csv", index=False)
segments.to_csv(out_dir / f"{trip_id}_segments.csv", index=False)
alertness_stats.to_csv(out_dir / "alertness_stats.csv")
out_dir